# 4.6 Lab - Use a Databricks Default DAB Template
**Lesson:** DAB Project Templates Overview

`databricks bundle init` scaffolds a complete project from a template:

| Template | What you get |
|---|---|
| `default-python` | Python package + notebook + job + (optional) Lakeflow pipeline, tests, `pyproject.toml` |
| `default-sql` | `.sql` files run as job tasks on a SQL warehouse |
| `default-minimal` | Only `databricks.yml`. For people who know what they want |
| `pydabs` | Like default-python, but resources are defined **in Python** instead of YAML |
| `dbt-sql`, `mlops-stacks`, `default-scala` | dbt projects, the full MLOps reference architecture, Scala JARs |
| your own | Any folder or Git repository with a `databricks_template_schema.json`. This is how platform teams standardize projects |

`bundle init` needs a logged-in workspace (it reads your user name and the workspace features). The project is created in `.lab-data/dabs/` (git-ignored).

In [ ]:
import json
from adelab import bundles
from adelab.spark import DATA_ROOT

status = bundles.workspace_status()
OUT = DATA_ROOT / "dabs"
OUT.mkdir(parents=True, exist_ok=True)

# Answers to the template's questions (otherwise it asks interactively). Unanswered questions use the
# template defaults. Run `databricks bundle init default-python` in a terminal to see all questions.
config = {"project_name": "my_default_project"}
(OUT / "default-python.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))

## Step 1 - Initialize the project
![bundle init default-python turns template answers into a project tree with dev and prod targets](../images/4.6-default-python-template.svg)

**How to read it:** the JSON written above answers the template questions, so `bundle init` runs without prompts. The middle box is the
kind of tree you will explore in Step 2: `databricks.yml` with a dev and a prod target, resources, a Python package,
and tests. Your CLI version decides the exact file names, so compare with `bundles.tree()`.

In [ ]:
PROJECT = OUT / "my_default_project"
if bundles.require_workspace(status) and not PROJECT.exists():
    bundles.cli("bundle", "init", "default-python", "--config-file", str(OUT / "default-python.json"),
                "--output-dir", str(OUT))

## Step 2 - Explore what was generated
Find answers to these questions in the generated files:
1. Which **targets** exist, and how does `prod` differ from `dev`?
2. Where is the job defined, which tasks does it have, and on which compute do they run?
3. How does the **wheel** get built? (look for `artifacts`)
4. Where are the tests, and how do they get a Spark session? (*Databricks Connect*: the tests run locally but execute on serverless compute)

In [ ]:
if PROJECT.exists():
    bundles.tree(PROJECT)
    print((PROJECT / "databricks.yml").read_text())

## Step 3 - Validate, deploy, run, destroy

In [ ]:
if PROJECT.exists() and bundles.require_workspace(status):
    bundles.cli("bundle", "validate", cwd=PROJECT)
    bundles.cli("bundle", "deploy", cwd=PROJECT)
    bundles.cli("bundle", "summary", cwd=PROJECT)

In [ ]:
if PROJECT.exists() and bundles.require_workspace(status):
    jobs = list(bundles.preview(PROJECT, "dev", show=False)["resources"].get("jobs", {}))
    bundles.cli("bundle", "run", jobs[0], cwd=PROJECT)

In [ ]:
if PROJECT.exists() and bundles.require_workspace(status):
    bundles.cli("bundle", "destroy", "--auto-approve", cwd=PROJECT)

## Step 4 (optional) - Try `pydabs`
Initialize the `pydabs` template next to it and compare: the job is a Python object (`databricks.bundles.jobs.Job`), so you can generate resources with loops and functions, much like the multiplex metaprogramming in Course 1.